# Python Web Scraper Comparison: BeautifulSoup vs Scrapy vs Crawl4AI

This notebook compares three popular Python web scraping tools. We focus on the use case of **extracting content for AI Agents and LLMs**, where clean Markdown is the gold standard.

### The Contenders

1. **BeautifulSoup**: The classic choice for simple, static HTML parsing.
2. **Scrapy**: The heavy-lifter framework for large-scale scraping projects.
3. **Crawl4AI**: The modern, AI-first tool that extracts clean Markdown automatically.


## 1. BeautifulSoup

**Pros**: Easy to learn, great for simple tasks.
**Cons**: Requires manual parsing logic; doesn't handle JavaScript; no built-in Markdown conversion.


In [5]:
import requests
from bs4 import BeautifulSoup


def bs4_example(url):
    # 1. Fetch the page (requires external library like requests)
    response = requests.get(url)

    # 2. Parse HTML
    soup = BeautifulSoup(response.content, "html.parser")

    # 3. Manual Extraction (The hard part)
    # You must manually find tags and clean text.
    # Converting this to Markdown requires writing a custom converter.
    title = soup.title.string if soup.title else "No Title"
    paragraphs = [p.get_text() for p in soup.find_all("p")]

    return f"Title: {title}\nContent Preview: {' '.join(paragraphs)[:200]}..."


print(bs4_example("https://example.com"))

Title: Example Domain
Content Preview: This domain is for use in documentation examples without needing permission. Avoid use in operations.Learn more Learn more...


## 2. Scrapy

**Pros**: Extremely powerful, fast, handles concurrency well.
**Cons**: Steep learning curve; boilerplate heavy; overkill for single scripts; Markdown extraction still requires custom logic.


In [ ]:
import scrapy
from scrapy.crawler import CrawlerProcess
import sys


# Scrapy requires defining a Class and often a full project structure
class DemoSpider(scrapy.Spider):
    name = "demo"
    start_urls = ["https://example.com"]

    def parse(self, response):
        # Still requires CSS/XPath selectors
        yield {"title": response.css("title::text").get(), "text": response.css("p::text").getall()}


# NOTE: Scrapy uses the Twisted reactor which conflicts with Jupyter's asyncio event loop.
# Running process.start() here will cause: "RuntimeError: This event loop is already running"
# This code is best run as a standalone script (e.g., `python spider.py`).

print("Scrapy code structure defined above.")
print("Skipping execution in notebook to avoid Reactor/EventLoop conflicts.")

# To run this in a standard script:
process = CrawlerProcess()
process.crawl(DemoSpider)
process.start()

2026-01-13 19:58:52 [scrapy.utils.log] INFO: Scrapy 2.14.1 started (bot: scrapybot)
2026-01-13 19:58:52 [scrapy.utils.log] INFO: Versions:
{'lxml': '5.4.0',
 'libxml2': '2.13.8',
 'cssselect': '1.3.0',
 'parsel': '1.10.0',
 'w3lib': '2.3.1',
 'Twisted': '25.5.0',
 'Python': '3.12.9 | packaged by Anaconda, Inc. | (main, Feb  6 2025, '
           '12:55:12) [Clang 14.0.6 ]',
 'pyOpenSSL': '25.3.0 (OpenSSL 3.5.4 30 Sep 2025)',
 'cryptography': '46.0.3',
 'Platform': 'macOS-26.2-arm64-arm-64bit'}
2026-01-13 19:58:52 [scrapy.crawler] DEBUG: Using CrawlerProcess
2026-01-13 19:58:52 [scrapy.addons] INFO: Enabled addons:
[]
2026-01-13 19:58:52 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2026-01-13 19:58:52 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.unix_events._UnixSelectorEventLoop
2026-01-13 19:58:52 [scrapy.extensions.telnet] INFO: Telnet Password: d147314beba3081e
2026-01-13 19:58:52 [scrapy.middleware] INFO: Enabled exte

Scrapy code structure defined above.
Skipping execution in notebook to avoid Reactor/EventLoop conflicts.


RuntimeError: This event loop is already running

2026-01-13 19:58:52 [scrapy.core.engine] INFO: Spider opened
2026-01-13 19:58:52 [scrapy.extensions.logstats] INFO: Crawled 0 pages (at 0 pages/min), scraped 0 items (at 0 items/min)
2026-01-13 19:58:52 [scrapy.extensions.telnet] INFO: Telnet console listening on 127.0.0.1:6023
2026-01-13 19:58:52 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://example.com> (referer: None)
2026-01-13 19:58:52 [scrapy.core.scraper] DEBUG: Scraped from <200 https://example.com>
{'title': 'Example Domain', 'text': ['This domain is for use in documentation examples without needing permission. Avoid use in operations.']}
2026-01-13 19:58:52 [scrapy.core.engine] INFO: Closing spider (finished)
2026-01-13 19:58:52 [scrapy.statscollectors] INFO: Dumping Scrapy stats:
{'downloader/request_bytes': 222,
 'downloader/request_count': 1,
 'downloader/request_method_count/GET': 1,
 'downloader/response_bytes': 646,
 'downloader/response_count': 1,
 'downloader/response_status_count/200': 1,
 'elapsed_time_seco

: 

## 3. Crawl4AI (The Winner for AI)

**Pros**:

- **Native Markdown**: Extracts clean Markdown ready for LLMs immediately.
- **Smart**: Handles dynamic content (JS) automatically.
- **Simple**: Minimal code for maximum output.
- **Free & Open Source**: No API keys required.

Crawl4AI abstracts away the browser management and HTML-to-Markdown conversion, making it the perfect tool for feeding data to AI.


In [ ]:
import asyncio
from crawl4ai import AsyncWebCrawler


async def crawl4ai_example():
    # Context manager handles browser session automatically
    async with AsyncWebCrawler(verbose=True) as crawler:
        # One line to fetch, render JS, and convert to Markdown
        result = await crawler.arun(url="https://www.nbcnews.com/business")

        # The Magic: result.markdown
        # No manual parsing needed. It's ready for your LLM prompt.
        print(f"--- Extracted Markdown (Length: {len(result.markdown)}) ---")
        print(result.markdown[:4000])  # Preview first 1000 chars


# In a Jupyter Notebook, you can await directly or use asyncio.run()
await crawl4ai_example()

[INIT].... → Crawl4AI 0.7.8 

[FETCH]... ↓ https://www.nbcnews.com/business                                                                     |
✓ | ⏱: 1.99s 

[SCRAPE].. ◆ https://www.nbcnews.com/business                                                                     |
✓ | ⏱: 0.05s 

[COMPLETE] ● https://www.nbcnews.com/business                                                                     |
✓ | ⏱: 2.04s 

--- Extracted Markdown (Length: 34215) ---
IE 11 is not supported. For an optimal experience visit our site on another browser.
Skip to Content
[NBC News Logo](https://www.nbcnews.com)
Sponsored By
  * [ Politics](https://www.nbcnews.com/politics)
  * [ U.S. News](https://www.nbcnews.com/us-news)
  * [ World](https://www.nbcnews.com/world)
  * Local
    *       * [New York](https://www.nbcnews.com/new-york)
      * [Los Angeles](https://www.nbcnews.com/los-angeles)
      * [Chicago](https://www.nbcnews.com/chicago)
      * [Dallas-Fort Worth](https://www.nbcnews.com/dallas-fort-worth)
      * [Philadelphia](https://www.nbcnews.com/philadelphia)
      * [Washington, D.C.](https://www.nbcnews.com/washington)
      * [Boston](https://www.nbcnews.com/boston)
      * [Bay Area](https://www.nbcnews.com/bay-area)
      * [South Florida](https://www.nbcnews.com/miami)
      * [San Diego](https://www.nbcnews.com/san-diego)
      * [Connecticut](https://www.nbcnews.com/connecticut)
  * [ Sports]

## Summary

| Feature                  | BeautifulSoup       | Scrapy               | Crawl4AI             |
| ------------------------ | ------------------- | -------------------- | -------------------- |
| **Complexity**           | Low                 | High                 | Low                  |
| **Dynamic Content (JS)** | No                  | No (Native)          | **Yes**              |
| **Output Format**        | HTML/Text           | Structured Data      | **Markdown**         |
| **Best For**             | Simple Static Pages | Large Scale Crawling | **AI Agents / LLMs** |
